# Fixed Next Word Predictor

This notebook contains the corrected next-word prediction code to work with the model trained in `new.ipynb`.

### Key Fixes Implemented:
1. **Direct $O(1)$ Word Lookup**: Replaced the extremely slow sequential loop over `tokenizer.word_index` with a direct dictionary lookup using `tokenizer.index_word.get(pos, "")` which is $O(1)$ complexity.
2. **Handling Unknown/Padding Tokens**: Prevented the model from predicting `<UNK>` (index 1) or padding (index 0) by zeroing out their probabilities before selecting the maximum probability token. This prevents the loop from immediately breaking.
3. **Restricting Predictions to the Active Vocabulary**: Since the tokenizer was initialized with `num_words=5000`, the sequences only contained token indices `< 5000`. The output classes `>= 5000` had untrained weights that outputted random noise. We slice the prediction array to `[:MAX_WORDS]` to only consider the trained vocabulary range.
4. **Portability**: Replaced `X.shape[1]` with `model.input_shape[1]` so that the sequence length is dynamically read from the model metadata.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model

# 1. Load the original text corpus to refit the tokenizer
with open("wikitext_10M.txt", "r", encoding="utf-8") as f:
    faqs = f.read()

print("Corpus length (chars):", len(faqs))

FileNotFoundError: [Errno 2] No such file or directory: 'wikitext_10M.txt'

In [ ]:
# 2. Initialize and fit the tokenizer exactly as done during training
MAX_WORDS = 5000

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<UNK>"
)

tokenizer.fit_on_texts(faqs.split('\n'))
tokenizer.fit_on_texts([faqs])

print("Total unique words in vocabulary:", len(tokenizer.word_index))

In [ ]:
# 3. Load the pre-trained model
model = load_model("best_next_word.keras")
model.summary()

In [ ]:
# 4. Run the fixed next-word prediction loop
text = "Artificial Intelligence"
maxlen = model.input_shape[1]  # Get sequence length (20 in this case)

print(f"Generating next words for: '{text}'...\n")

for i in range(24):
    # Convert input text to sequences
    token_text = tokenizer.texts_to_sequences([text])[0]
    
    # Pad the sequence to match the model's input shape
    padded_token_text = pad_sequences([token_text], maxlen=maxlen, padding='pre')

    # Predict next-word probabilities
    pred = model.predict(padded_token_text, verbose=0)

    # Slices to only consider the active training vocabulary (top 5000 words)
    pred_restricted = pred[0][:MAX_WORDS]
    
    # Zero out the padding (0) and UNK (1) probabilities so they are not predicted
    pred_restricted[0] = 0.0
    pred_restricted[1] = 0.0

    # Choose the class with highest probability from the filtered subset
    pos = np.argmax(pred_restricted)
    
    # Direct O(1) dictionary lookup
    output_word = tokenizer.index_word.get(pos, "")

    # Break only if we failed to retrieve any word
    if output_word == "":
        break

    text += " " + output_word

print("Final Text Output:")
print(text)